# tau-chrono Quickstart

**Bayesian Noise Tracker for Quantum Circuits**

This notebook demonstrates the core capabilities of tau-chrono in 5 minutes, with zero hardware required.

**What you'll learn:**
1. How to define noise channels
2. How Bayesian composition improves noise estimation
3. How to interpret per-gate tau values
4. How to visualize results

In [ ]:
# Install tau-chrono (uncomment if needed)
# !pip install tau-chrono[viz]

import numpy as np
from tau_chrono import (
    amplitude_damping,
    depolarizing,
    dephasing,
    bayesian_compose,
    verify_cptp,
)

print(f"tau-chrono imported successfully!")

## Step 1: Define noise channels

Every gate on a real quantum computer introduces noise. tau-chrono models each gate's noise as a quantum channel (list of Kraus operators).

In [ ]:
# Define a 5-gate noise sequence (typical single-qubit circuit)
channels = [
    amplitude_damping(0.05),   # T1 decay
    depolarizing(0.08),        # depolarizing noise
    amplitude_damping(0.12),   # stronger T1 decay
    dephasing(0.03),           # T2 dephasing
    amplitude_damping(0.10),   # more T1 decay
]
names = ["AD(0.05)", "Dep(0.08)", "AD(0.12)", "Deph(0.03)", "AD(0.10)"]

# Verify all channels are valid CPTP maps
for name, ch in zip(names, channels):
    ok, err = verify_cptp(ch)
    print(f"{name}: CPTP={ok}, error={err:.2e}")

## Step 2: Run Bayesian composition

The key insight: instead of treating each gate's noise independently (which overestimates total error), tau-chrono **propagates the reference state** through the circuit. Each gate gets the correct Bayesian prior from its predecessor.

In [ ]:
# Input state |+> and reference state
rho   = 0.5 * np.array([[1, 1], [1, 1]], dtype=complex)
sigma = np.diag([0.8, 0.2]).astype(complex)

# Run Bayesian analysis
result = bayesian_compose(channels, sigma, rho, channel_names=names)

# Summary
print(f"Total tau (naive/multiplicative): {result.tau_multiplicative_total:.4f}")
print(f"Total tau (Bayesian):             {result.tau_bayesian_total:.4f}")
print(f"Improvement:                      {result.improvement_percent:.1f}%")
print(f"Composition inequality holds:     {result.composition_holds}")

## Step 3: Inspect per-gate results

Each gate gets a `tau_naive` (independent estimate) and `tau_eff` (Bayesian estimate). The difference grows with circuit depth.

In [ ]:
print(f"{'Gate':<12} {'tau_naive':>10} {'tau_eff':>10} {'Class':>18}")
print("-" * 55)
for gr in result.gate_results:
    print(f"{gr.channel_name:<12} {gr.tau_naive:>10.6f} {gr.tau_eff:>10.6f} {gr.classification:>18}")

## Step 4: Depth scaling comparison

Let's see how Bayesian tracking scales vs the naive approach at increasing circuit depth.

In [ ]:
depths = [2, 4, 6, 8, 10, 15, 20, 30]
tau_naive_list = []
tau_bayes_list = []

for d in depths:
    chs = [depolarizing(0.08)] * d
    r = bayesian_compose(chs, sigma, rho)
    tau_naive_list.append(r.tau_multiplicative_total)
    tau_bayes_list.append(r.tau_bayesian_total)

print(f"{'Depth':>5} {'tau_naive':>10} {'tau_Bayes':>10} {'Improve':>10}")
print("-" * 40)
for d, tn, tb in zip(depths, tau_naive_list, tau_bayes_list):
    imp = 100.0 * (tn - tb) / tn if tn > 0 else 0
    print(f"{d:>5} {tn:>10.4f} {tb:>10.4f} {imp:>9.1f}%")

## Step 5: Visualize (requires matplotlib)

tau-chrono includes built-in visualization for quick diagnostics.

In [ ]:
try:
    from tau_chrono import plot_tau_heatmap, plot_depth_scaling

    # Per-gate tau heatmap
    fig1 = plot_tau_heatmap(result)
    fig1.set_size_inches(8, 4)
    display(fig1)

    # Depth scaling plot
    fig2 = plot_depth_scaling(depths, tau_naive_list, tau_bayes_list)
    fig2.set_size_inches(8, 5)
    display(fig2)

except ImportError:
    print("Install matplotlib for visualization: pip install tau-chrono[viz]")

## Next steps

- **Hardware validation**: Connect to IBM Quantum or Quantum Inspire with `tau-chrono[qiskit]`
- **2-qubit analysis**: Use `two_qubit_depolarizing`, `cnot_error` for multi-qubit circuits
- **Error mitigation**: Combine with `tau_chrono.mitigation.tau_informed_zne`
- **CLI**: Run `tau-chrono analyze "ad:0.05,dep:0.08"` from the terminal

See the full docs at https://tau-chrono.readthedocs.io